# ukpyn — Python client for UK Power Networks Open Data

`pip install ukpyn`

## First query — three lines of code

In [24]:
from ukpyn import ltds

In [25]:
# Get observed peak demand at every primary substation
observed = ltds.get_table_3a(licence_area="SPN", limit=5)

print(f"Total records available: {observed.total_count}")
print(f"Returned: {len(observed.records)}")
print()

# Show the first record
rec = observed.records[0]
for key, val in sorted(rec.fields.items()):
    print(f"  {key}: {val}")

Total records available: 596
Returned: 5

  firm_capacity_mw: 22.5
  forecast_m_d_mw_25_26: 7.0
  forecast_m_d_mw_26_27: 7.0
  forecast_m_d_mw_27_28: 6.9
  forecast_m_d_mw_28_29: 7.2
  forecast_m_d_mw_29_30: 7.4
  functional_location: SPN-S000000008609
  gridsupplypoint: Beddington
  licencearea: South Eastern Power Networks (SPN)
  maximum_demand_24_25_mw: 6.3
  maximum_demand_24_25_pf: 0.98
  minimum_load_scaling_factor: None
  season: Winter
  substation: Addington Grid 11kV
  unutilised_capacity_percent: 72.0


---

## 3. Orchestrators

"ukpyn groups datasets into orchestrators — one for each domain."



## Summary

| Strategy question | Key datasets | Orchestrator |
|---|---|---|
| Is there capacity for heat pumps / EVs? | `headroom`, `lct_lsoa`, `lct_secondary` | `dfes` |
| What's the current peak demand? | `table_3a`, `table_3b` | `ltds` |
| What generation is already connected? | `ecr`, `ecr_small`, `table_5` | `ders`, `ltds` |
| Where are the substations? | `grid_primary_sites`, `secondary_sites` | `gis` |
| What reinforcement is planned? | `infrastructure_projects` | `ltds` |
| Which areas are constrained? | `dnoa`, `headroom` (negative values) | `dnoa`, `dfes` |
| Where should EV chargers go? | `chargepoints`, `headroom`, `by_local_authority` | `dfes` + client |
| What does the network look like on a map? | `hv_overhead_lines`, `licence_boundaries` + `export_geojson()` | `gis` |

For more details, see the [full documentation](https://ukpn-dso.github.io/ukpyn/) and the other [tutorials](README.md).

In [26]:
from ukpyn import ders, dfes, gis, ltds
from ukpyn.registry import ALL_DATASETS

print(f"ukpyn knows {len(ALL_DATASETS)} dataset aliases across 5 orchestrators:")
print()
print("  ltds  — Long Term Development Statement (network planning)")
print("  dfes  — Distribution Future Energy Scenarios (forecasts)")
print("  ders  — Distributed Energy Resources (generation register)")
print("  gis   — Geospatial (substations, lines, boundaries)")
print("  dnoa  — Distribution Network Options Assessment (flex vs build)")

ukpyn knows 118 dataset aliases across 5 orchestrators:

  ltds  — Long Term Development Statement (network planning)
  dfes  — Distribution Future Energy Scenarios (forecasts)
  ders  — Distributed Energy Resources (generation register)
  gis   — Geospatial (substations, lines, boundaries)
  dnoa  — Distribution Network Options Assessment (flex vs build)


## Filter to a Local Authority

In [27]:
# Embedded generation ≥1MW in Mole Valley
ecr = ders.get(
    "embedded_capacity",
    limit=10,
    where="local_authority = 'Mole Valley'",
)

print(f"Generation sites in Mole Valley: {ecr.total_count}")
print()

for rec in ecr.records:
    f = rec.fields
    source = f.get("energy_source_1") or "?"
    capacity = f.get("maximum_export_capacity_mw") or "?"
    status = f.get("connection_status") or "?"
    print(f"  {source:20s} | {str(capacity):>8} MW | {status}")

Generation sites in Mole Valley: 6

  Solar                |      3.4 MW | Connected
  Solar                |        ? MW | Accepted to Connect
  Fossil - Gas         |      1.1 MW | Connected
  Stored Energy (all stored energy irrespective of the original energy source) |        ? MW | Accepted to Connect
  Stored Energy (all stored energy irrespective of the original energy source) |      6.0 MW | Connected
  Solar                |        ? MW | Accepted to Connect


In [28]:
# DFES future scenarios for Mole Valley
la = dfes.get(
    "by_local_authority",
    limit=5,
    where="lad22nm = 'Mole Valley'",
)

print(f"DFES scenario records for Mole Valley: {la.total_count}")
print()

for rec in la.records[:5]:
    f = rec.fields
    pathway = f.get("pathway", "?")
    year = f.get("year", "?")
    hps = f.get("number_of_domestic_heat_pumps", "?")
    evs = f.get("number_of_electric_cars_hybrid_and_full_electric", "?")
    print(f"  {pathway:20s} | {year} | Heat pumps: {hps:>8} | EVs: {evs:>8}")

DFES scenario records for Mole Valley: 6048

  Holistic Transition  | 2032 | Heat pumps:       14 | EVs:      545
  Holistic Transition  | 2033 | Heat pumps:       15 | EVs:      611
  Holistic Transition  | 2034 | Heat pumps:       17 | EVs:      673
  Holistic Transition  | 2039 | Heat pumps:      115 | EVs:      908
  Holistic Transition  | 2042 | Heat pumps:      170 | EVs:      994


## Geospatial — every record has `.geometry`

In [29]:
# Show geometry on individual records
substations = gis.get_primary_substations(licence_area="SPN", limit=5)

print(f"Primary substations (SPN): {substations.total_count}")
print()

for rec in substations.records[:3]:
    name = rec.fields.get("sitefunctionallocation", "?")
    print(f"  {str(name):30s} → {rec.geometry}")

Primary substations (SPN): 270

  SPN-S000000008020              → {'type': 'Point', 'coordinates': [1.126801, 51.088393]}
  SPN-S000000008025              → {'type': 'Point', 'coordinates': [-0.362501, 51.422094]}
  SPN-S000000008129              → {'type': 'Point', 'coordinates': [0.434154, 50.84336]}


In [30]:
import json

# Export licence area boundaries as 2D GeoJSON
geojson_bytes = gis.export_geojson("licence_boundaries", dimensions="2d")
geojson = json.loads(geojson_bytes)

print(f"GeoJSON type: {geojson['type']}")
print(f"Features: {len(geojson['features'])}")
print()

for feat in geojson["features"]:
    props = feat.get("properties", {})
    geom_type = feat["geometry"]["type"]
    print(f"  {props.get('name', '?'):30s} — {geom_type}")

# write to file to Copy this to clipboard, paste into geojson.io
with open("licence_boundaries.geojson", "w") as f:
    json.dump(geojson, f)

GeoJSON type: FeatureCollection
Features: 3

  EPN                            — Polygon
  LPN                            — Polygon
  SPN                            — Polygon


## Data quality — handled automatically

In [31]:
import json

rec = substations.records[0]

# This would fail with raw ODP data if NaN values were present
json_str = json.dumps(rec.fields)
print("✓ JSON serialisation works (NaN values sanitised to None)")
print(f"✓ Normalised geometry: {rec.geometry}")
print("✓ Feature wrappers unwrapped automatically")
print("✓ 2D/3D coordinate control via dimensions parameter")

✓ JSON serialisation works (NaN values sanitised to None)
✓ Normalised geometry: {'type': 'Point', 'coordinates': [1.126801, 51.088393]}
✓ Feature wrappers unwrapped automatically
✓ 2D/3D coordinate control via dimensions parameter


## 133+ datasets in one library

In [32]:
from ukpyn.registry import (
    CONNECTIONS_DATASETS,
    CURTAILMENT_DATASETS,
    DER_DATASETS,
    DFES_DATASETS,
    DNOA_DATASETS,
    EQUIPMENT_DATASETS,
    FLEXIBILITY_DATASETS,
    GEO_DATASETS,
    LTDS_DATASETS,
    POWERFLOW_DATASETS,
    REFERENCE_DATASETS,
    SMART_LCT_DATASETS,
)

groups = {
    "LTDS (network planning)": LTDS_DATASETS,
    "DFES (future scenarios)": DFES_DATASETS,
    "DNOA (flex vs build)": DNOA_DATASETS,
    "GIS (geospatial)": GEO_DATASETS,
    "DER (generation)": DER_DATASETS,
    "Flexibility & Curtailment": {**FLEXIBILITY_DATASETS, **CURTAILMENT_DATASETS},
    "Powerflow (time series)": POWERFLOW_DATASETS,
    "Smart Meters & LCT": SMART_LCT_DATASETS,
    "Connections": CONNECTIONS_DATASETS,
    "Equipment": EQUIPMENT_DATASETS,
    "Reference": REFERENCE_DATASETS,
}

total = 0
for name, datasets in groups.items():
    unique = set(datasets.values())
    total += len(unique)
    print(f"  {name:35s}  {len(unique):>3} datasets")

print(f"\n  {'TOTAL':35s}  {total:>3} datasets")

  LTDS (network planning)               14 datasets
  DFES (future scenarios)                3 datasets
  DNOA (flex vs build)                   2 datasets
  GIS (geospatial)                      19 datasets
  DER (generation)                       4 datasets
  Flexibility & Curtailment              3 datasets
  Powerflow (time series)               10 datasets
  Smart Meters & LCT                     6 datasets
  Connections                            7 datasets
  Equipment                              5 datasets
  Reference                              5 datasets

  TOTAL                                 78 datasets


## More info

- **Docs:** https://ukpn-dso.github.io/ukpyn/
- **PyPI:** https://pypi.org/project/ukpyn/
- **Install:** `pip install ukpyn`